# Task 3 response playground

Task 3 is easy to misread: a mutant can still be very close to wild type in absolute gene expression, so a prediction can look plausible without actually capturing the **effect of the knockout**.

This notebook uses the small matched-WT / Mab21l2 examples published with the organizers' `veckit` scorer and breaks the known perturbation in a few controlled ways. We will make the response too weak, too strong, reversed, or attached to the wrong genes, then watch the official metrics change.

The goal is not to solve Gata4 or β-catenin. The Mab21l2 target is known on purpose. That makes this a scorer/debugging experiment rather than a held-out prediction experiment.

The bundled examples are only about 150 cells, so they run quickly but are noisy and are **not valid competition submissions**.


## What are we trying to learn?

There are several different ways a perturbation prediction can be wrong.

- **DES** asks whether the genes that should respond are actually recovered.
- **DCS** asks whether those changes go in the right direction.
- **severity_slope** asks whether the response is roughly the right size.
- **MMD and the variogram/CSS metric** look at the generated cell population rather than only its average.

Instead of treating those as five definitions to memorize, we will make one known mistake at a time and see which numbers react.


In [ ]:
# Install the public scorer at the commit this notebook was checked against.
!pip -q install "git+https://github.com/aristoteleo/veckit.git@46d41e63f42a9aab815db20b742feeccd249cb17" pandas matplotlib


In [ ]:
# Download only the tiny public examples distributed with the same veckit commit.
from pathlib import Path
import urllib.request

DATA = Path('/content/vec_t3_playground')
DATA.mkdir(exist_ok=True)
base = 'https://raw.githubusercontent.com/aristoteleo/veckit/46d41e63f42a9aab815db20b742feeccd249cb17/data/'
for name in ['sample_wt.h5ad', 'sample_mab21l2_ko.h5ad']:
    p = DATA / name
    if not p.exists():
        urllib.request.urlretrieve(base + name, p)
    print(name, round(p.stat().st_size/1e6, 2), 'MB')


In [ ]:
import anndata as ad
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import sparse
from veckit import score

wt_path = DATA / 'sample_wt.h5ad'
ko_path = DATA / 'sample_mab21l2_ko.h5ad'
wt = ad.read_h5ad(wt_path)
ko = ad.read_h5ad(ko_path)

assert list(wt.var_names) == list(ko.var_names), 'Sample panels differ'
print('WT:', wt.shape, 'KO:', ko.shape)
print('WT has spatial_3D:', 'spatial_3D' in wt.obsm)
print('KO has spatial_3D:', 'spatial_3D' in ko.obsm)


In [ ]:
def dense_X(a):
    return a.X.toarray().astype(np.float32) if sparse.issparse(a.X) else np.asarray(a.X, dtype=np.float32)

X_wt = dense_X(wt)
X_ko = dense_X(ko)

# Population-level response. No cell correspondence is assumed.
delta_pb = X_ko.mean(axis=0) - X_wt.mean(axis=0)

print('genes:', len(delta_pb))
print('mean |pseudobulk shift|:', float(np.abs(delta_pb).mean()))
print('largest shifts:')
top = np.argsort(np.abs(delta_pb))[-10:][::-1]
for j in top:
    print(f'{wt.var_names[j]:>14s}  {delta_pb[j]: .4f}')


## Build a few deliberately simple predictions

First compute the average Mab21l2 response:

`known response = mean(KO) - mean(WT)`

Then add a scaled copy of that same response to every WT cell:

`prediction = clip(WT + α × known response, 0)`

This is deliberately crude. It is useful here *because* it isolates one variable at a time.

- `α = 0` means no knockout response at all.
- `0 < α < 1` has the right average direction but is too weak.
- `α = 1` matches the known pseudobulk shift.
- `α > 1` is too strong.
- `reversed` flips the sign of the response.
- `shuffled` keeps the same collection of response values but assigns them to the wrong genes.


In [ ]:
OUT = DATA / 'predictions'
OUT.mkdir(exist_ok=True)

def write_prediction(name, shift):
    pred = wt.copy()
    pred.X = np.clip(X_wt + shift[None, :], 0, None).astype(np.float32)
    p = OUT / f'{name}.h5ad'
    pred.write_h5ad(p)
    return p

preds = {}
for alpha in [0.0, 0.25, 0.5, 1.0, 1.5, 2.0]:
    preds[f'alpha_{alpha:g}'] = write_prediction(f'alpha_{alpha:g}', alpha * delta_pb)

preds['reversed'] = write_prediction('reversed', -delta_pb)

rng = np.random.default_rng(0)
shuffled = delta_pb[rng.permutation(len(delta_pb))]
preds['shuffled'] = write_prediction('shuffled', shuffled)

print('wrote', len(preds), 'controlled predictions')


In [ ]:
# Score each controlled prediction against the KNOWN training knockout.
# This is a metric probe only; it says nothing about held-out Gata4 / beta-catenin performance.
rows = []
for name, path in preds.items():
    result = score(task='T3', input=path, target=ko_path, wt=wt_path)
    m = result['metrics']
    pred = ad.read_h5ad(path)
    pb_pred = dense_X(pred).mean(axis=0)
    pb_true = X_ko.mean(axis=0)
    abs_pb_pearson = float(np.corrcoef(pb_pred, pb_true)[0, 1])
    rows.append({
        'prediction': name,
        'de_score': m.get('de_score'),
        'de_direction': m.get('de_direction'),
        'severity_slope': m.get('severity_slope'),
        'mmd_u': m.get('mmd_u'),
        'variogram': m.get('variogram'),
        'absolute_pb_pearson': abs_pb_pearson,
    })

df = pd.DataFrame(rows).set_index('prediction')
df.round(4)


In [ ]:
display(df.sort_values('de_direction', ascending=False).round(4))

ax = df[['de_score','de_direction','absolute_pb_pearson']].plot(marker='o', figsize=(10,5))
ax.set_title('Right state is not the same as right perturbation response')
ax.set_ylabel('metric value')
ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.show()


In [ ]:
sweep = df.loc[[f'alpha_{a:g}' for a in [0.0,0.25,0.5,1.0,1.5,2.0]]].copy()
sweep['alpha'] = [0.0,0.25,0.5,1.0,1.5,2.0]
sweep = sweep.set_index('alpha')

for metric in ['de_score','de_direction','severity_slope','mmd_u','variogram']:
    ax = sweep[metric].plot(marker='o', figsize=(7,4), title=f'{metric} as response strength is scaled')
    ax.axvline(1.0, linestyle='--', alpha=0.5)
    ax.set_xlabel('response scale α (1 = known Mab21l2 pseudobulk shift)')
    plt.tight_layout()
    plt.show()


## Reading the plots

With only ~150 cells, do not expect every line to be perfectly smooth. MMD and the variogram are especially noisy at this scale. The useful checks are qualitative.

1. The no-change prediction can still have a high absolute expression correlation because most of the embryo did not change very much. That does **not** mean the perturbation was predicted correctly.
2. Reversing the response should hurt direction-sensitive metrics.
3. Shuffling the response asks whether the scorer can distinguish “genes changed” from “the *right* genes changed.”
4. Sweeping `α` separates response direction from response strength.
5. A simple uniform shift can get the average response roughly right while still producing the wrong single-cell distribution. That is what the distributional metrics are there to catch.


## Sources

- Official Task 3 description: https://virtualembryo.ai/challenge/tasks/perturbation
- Official Task 3 metric definitions: https://virtualembryo.ai/challenge/evaluation?section=scoring&task=3
- Organizers' local scorer and tiny public samples: https://github.com/aristoteleo/veckit

Independent community notebook. No hidden challenge data is included or accessed.
